# Real-world data coding for neuroscientists (ReCoN)

### MSc in Translational Neuroscience,

### Department of Brain Sciences, Faculty of Medicine,

### Imperial College London

### Contributors: Anastasia Ilina, Jared Maina, Cecilia Rodriguez, Marirena Bafaloukou, Katarzyna Marta Zoltowska, Rishideep Chatterjee, Sahar Rahbar, Cynthia Sandor

### Autumn 2026

---
## Tutorial 2 – Tabular data with pandas on Electronic Healthcare Records (EHR)

### **Part 2: Statistical Analysis**

---

## Learning outcomes

**Overall aim**: run and interpret basic statistical tests on medical data in Python.

This notebook is about *choosing and running* tests, not about the mathematics behind them. **You will learn to:**
- Explain what a hypothesis test, a *p*-value and a confidence interval tell you, and what they don't.
- Check a test's assumptions and choose between parametric and non-parametric tests.
- Compare groups, measure associations and test categorical data with `scipy.stats`.
- Correct for multiple comparisons.
- Report a result in one clear sentence.

Each section ends with a ✅ **checkpoint**. If you can tick everything off, move on; if not, revisit that section first.

| Section | Key tools |
| --- | --- |
| 1. What a test does | permutation with `numpy`, `groupby` |
| 2. Choosing a test | Q–Q plots, `shapiro`, `levene` |
| 3. Comparing groups | `ttest_ind`, `mannwhitneyu`, `f_oneway`, `kruskal` |
| 4. Associations | `pearsonr`, `spearmanr`, `.corr`, `pd.crosstab`, `chi2_contingency` |
| 5. Multiple comparisons | `itertools.combinations`, `multipletests` |

---
## 📚 Supporting materials

> - [Learning Statistics with Python](https://ethanweed.github.io/pythonbook/landingpage.html): a free, beginner-friendly textbook
> - [The nuts and bolts of hypothesis testing](https://pmc.ncbi.nlm.nih.gov/articles/PMC6668281/)
> - [📉 `scipy.stats` reference](https://docs.scipy.org/doc/scipy/reference/stats.html): every test used here
> - [📊 Statsmodels user guide](https://www.statsmodels.org/stable/user-guide.html)

---
## Setup

Run the cells below. The first imports the libraries and defines `fmt_p()`, a small helper that prints p-values the way papers report them (very small values as *p* < 0.001). The second builds the same synthetic Parkinson's dataset as the previous notebook, this time with some real group differences built in. You don't need to read the generation code.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")

def fmt_p(p):
    # format a p-value for reporting: very small values become "p < 0.001"
    return "p < 0.001" if p < 0.001 else f"p = {p:.3f}"

In [ ]:
# === RUN THIS CELL TO (RE)CREATE THE SYNTHETIC DATASET ===
# To rebuild from scratch: run `del df`, then run this cell again.
expected_cols = {"LEDD","sex","years_since_dx","age","updrs_iii","patient_id","ethnicity","BMI"}

# Only (re)create if df isn't already present with expected columns
if not (isinstance(globals().get("df"), pd.DataFrame) and expected_cols.issubset(df.columns)):
    rng = np.random.default_rng(0)
    n = 5000

    # ---- Core table ----
    df = pd.DataFrame({
        "patient_id": [f"P{i:04d}" for i in range(n)],
        "sex": rng.choice(["Male","Female"], size=n, p=[0.55, 0.45]),
        "years_since_dx": rng.uniform(0, 12, size=n).round(2),
        "age": rng.normal(62, 8, size=n).round(1),
        "ethnicity": rng.choice(["White", "Black", "Asian", "Other", ""],
                                size=n, p=[0.70, 0.10, 0.10, 0.05, 0.05]),
        "BMI": rng.normal(27, 4, size=n).round(1),
    })

    # ---- LEDD: base + exponential noise (skewed) ----
    df["LEDD"] = (250 + 50*df["years_since_dx"] + rng.exponential(500, size=n)).clip(0).round(1)

    # ---- UPDRS III: realistic 0–132 with progression + heterogeneity ----
    # Patient-level random intercepts (baseline severity) and slopes (annual change)
    unique_ids = df["patient_id"].unique()
    intercepts = pd.Series(np.clip(rng.normal(20, 6, size=len(unique_ids)), 0, 60), index=unique_ids)
    slopes     = pd.Series(np.clip(rng.normal(2.2, 0.9, size=len(unique_ids)), 0, 5), index=unique_ids)

    alpha = df["patient_id"].map(intercepts).to_numpy()
    beta  = df["patient_id"].map(slopes).to_numpy()
    years = df["years_since_dx"].astype(float).to_numpy()
    eps   = rng.normal(0, 5, size=len(df))  # visit-level noise

    updrs = np.clip(alpha + beta*years + eps, 0, 132)
    df["updrs_iii"] = np.round(updrs, 1)

    # Inject a few impossible outliers (>132) for teaching Winsorisation/capping
    n_out = max(20, int(0.004 * len(df)))  # ~0.4% or at least 20 rows
    idx_out = rng.choice(df.index, size=n_out, replace=False)
    df.loc[idx_out, "updrs_iii"] = np.round(rng.uniform(140, 220, size=n_out), 1)



    # ---- Duplicate visits for some patients (longitudinal flavor) ----
    ###reps = df.sample(200, replace=True, random_state=0).copy()
    ##reps["patient_id"] = reps["patient_id"].sample(frac=1.0, random_state=1).values
    #df = pd.concat([df, reps], ignore_index=True)

    # make LEDD and UPDRS higher on average for men
    df.loc[df["sex"]=="Male", "LEDD"] *= rng.uniform(1.05, 1.15, size=(df["sex"]=="Male").sum())
    df.loc[df["sex"]=="Male", "updrs_iii"] *= rng.uniform(1.05, 1.15, size=(df["sex"]=="Male").sum())

    # Make LEDD and UPDRS higher with age and BMI o
    df["LEDD"] *= (1 + 0.03 * (df["age"] - df["age"].mean()))
    df["updrs_iii"] *= (1 + 0.02 * (df["age"] - df["age"].mean()))
    df["LEDD"] *= (1 + 0.02 * (df["BMI"] - df["BMI"].mean()))
    df["updrs_iii"] *= (1 + 0.01 * (df["BMI"] - df["BMI"].mean()))

    # Make LEDD and UPDRS higher in Black and Asian
    df.loc[df["ethnicity"]=="Black", "LEDD"] *= rng.uniform(1.10, 1.25, size=(df["ethnicity"]=="Black").sum())
    df.loc[df["ethnicity"]=="Asian", "LEDD"] *= rng.uniform(1.05, 1.15, size=(df["ethnicity"]=="Asian").sum())
    df.loc[df["ethnicity"]=="Black", "updrs_iii"] *= rng.uniform(1.10, 1.25, size=(df["ethnicity"]=="Black").sum())
    df.loc[df["ethnicity"]=="Other", "updrs_iii"] *= rng.uniform(1.05, 1.15, size=(df["ethnicity"]=="Other").sum())



    # ---- Intentional data-entry errors (for cleaning exercises) ----
    # BMI: ×10 typos and negatives
    df.loc[rng.choice(df.index, size=5, replace=False), "BMI"] *= 10
    df.loc[rng.choice(df.index, size=5, replace=False), "BMI"] *= -1
    # Age: negatives and unrealistic high values
    df.loc[rng.choice(df.index, size=5, replace=False), "age"] *= -1
    df.loc[rng.choice(df.index, size=5, replace=False), "age"] += 108


    # ---- MCAR missingness on selected columns ----
    for col, p in [("LEDD", 0.05), ("updrs_iii", 0.10), ("sex", 0.01), ("age", 0.02), ("ethnicity", 0.05)]:
        df.loc[rng.choice(df.index, size=int(p*len(df)), replace=False), col] = np.nan

    # ---- MNAR: LEDD more likely missing when UPDRS is high ----
    high_updrs = df["updrs_iii"] > 45
    # Safeguard: drop NaNs before sampling indices
    high_idx = df.index[high_updrs.fillna(False)]
    if len(high_idx) > 0:
        df.loc[rng.choice(high_idx, size=int(0.3*len(high_idx)), replace=False), "LEDD"] = np.nan

    df.reset_index(drop=True, inplace=True)

    # ---- Summary printout ----
    print(f"Created synthetic dataset with {len(df)} rows")
    print(f"Injected UPDRS>132 outliers: {(df['updrs_iii'] > 132).sum()}")
    print("Missingness (%):")
    print((df[["LEDD","updrs_iii","sex","age","ethnicity","BMI"]].isna().mean()*100).round(1))

    # ---- Duplicates in three buckets (exact, same-visit, legit repeat visits) ----
    # Counts: adjust as you like
    n_exact = 40          # exact row duplicates
    n_key   = 160         # same patient_id & years_since_dx (not exact)
    n_repeat= 300         # legit repeated visits (different years_since_dx)

    df_list = [df]

    # --- 1) Exact row duplicates: true copy-paste of entire rows ---
    idx_exact = rng.choice(df.index, size=n_exact, replace=False)
    exact_dups = df.loc[idx_exact].copy()
    df_list.append(exact_dups)

    # --- 2) Key duplicates: same patient_id & same years_since_dx but slight entry drift ---
    # pick base rows
    idx_key = rng.choice(df.index.difference(idx_exact), size=n_key, replace=False)
    key_dups = df.loc[idx_key].copy()

    # Introduce small, realistic data-entry drift to numeric fields (but keep ID + time the same)
    def small_perturb(x, scale):
        return np.round(x + rng.normal(0, scale, size=len(x)), 1)

    if "LEDD" in key_dups:
        key_dups["LEDD"] = small_perturb(key_dups["LEDD"], scale=15)     # ±15 mg
    if "updrs_iii" in key_dups:
        key_dups["updrs_iii"] = np.clip(small_perturb(key_dups["updrs_iii"], scale=1.5), 0, 220)
    if "BMI" in key_dups:
        key_dups["BMI"] = np.round(key_dups["BMI"] + rng.normal(0, 0.3, size=len(key_dups)), 1)

    # keep patient_id and years_since_dx unchanged (this makes them key duplicates)
    df_list.append(key_dups)

    # --- 3) Legit repeats: same patient, NEW years_since_dx (and recomputed fields) ---
    # choose patients to get an extra visit
    rep_patients = rng.choice(df["patient_id"].unique(), size=n_repeat, replace=False)

    # For each chosen patient, create a new visit at a later (or earlier) year_since_dx
    reps = df[df["patient_id"].isin(rep_patients)].copy()

    # Propose a new time that differs from the current one by ~0.5–2.0 years
    delta = rng.uniform(0.5, 2.0, size=len(reps))
    sign  = rng.choice([1, -1], size=len(reps), p=[0.8, 0.2])  # mostly later visits
    new_years = np.clip(reps["years_since_dx"].astype(float).to_numpy() + sign*delta, 0, 12)
    reps["years_since_dx"] = np.round(new_years, 2)

    # Age moves with time (if you want): comment out if not desired
    if "age" in reps:
        reps["age"] = np.round(reps["age"].astype(float) + sign*delta, 1)

    # Recompute LEDD and UPDRS for the *new* visit using the same subject-specific intercept/slope
    pid = reps["patient_id"]
    alpha_rep = pid.map(intercepts).to_numpy()
    beta_rep  = pid.map(slopes).to_numpy()
    years_rep = reps["years_since_dx"].astype(float).to_numpy()

    # fresh noise terms
    eps_rep = rng.normal(0, 5, size=len(reps))
    updrs_rep = np.clip(alpha_rep + beta_rep*years_rep + eps_rep, 0, 132)
    reps["updrs_iii"] = np.round(updrs_rep, 1)

    # recompute LEDD with new years_since_dx
    led_noise = rng.exponential(500, size=len(reps))
    reps["LEDD"] = (250 + 50*reps["years_since_dx"].astype(float) + led_noise).clip(0).round(1)

    # Append legit repeats
    df = pd.concat(df_list + [reps], ignore_index=True)

    # ---- Quick bookkeeping prints ----
    print("\nInjected duplicates summary:")
    print(f"  Exact duplicates added:         {len(exact_dups)}")
    print(f"  Key-duplicates added:           {len(key_dups)}")
    print(f"  Legit repeat visits added:      {len(reps)}")

    # (OPTIONAL) Sanity counts right now (pre-missingness/multipliers)
    exact_count = df.duplicated(keep=False).sum()
    key_count   = df.duplicated(subset=['patient_id','years_since_dx'], keep=False).sum()
    print(f"  Current exact-dup rows (counted individually): {exact_count}")
    print(f"  Current key-dup rows (same patient+time):      {key_count}")

### Cleaning recap: one row per patient

The tests in this notebook assume **independent observations**: each patient counts once. Counting the same patient twice makes a result look more certain than it is. Using the cleaning steps from the previous notebook, we:
1. drop exact duplicates;
2. keep each patient's **earliest visit**, so there is one row per patient;
3. set impossible values to `NaN`, and convert empty ethnicity strings to `NaN`.

In [ ]:
df_clean = (df.drop_duplicates()
              .sort_values(["patient_id", "years_since_dx"])   # each patient's earliest visit first
              .drop_duplicates(subset="patient_id", keep="first")   # then keep one row per patient
              .copy())
df_clean["ethnicity"] = df_clean["ethnicity"].replace("", np.nan)

plausible = {"age": (18, 110), "BMI": (10, 70), "updrs_iii": (0, 132)}
for col, (lo, hi) in plausible.items():
    df_clean[col] = df_clean[col].where(df_clean[col].between(lo, hi))   # .where keeps in-range values and sets the rest to NaN

print(f"{len(df)} visits → {len(df_clean)} patients")
df_clean.describe()

---
## 1. What does a statistical test do?

Do men and women have different mean LEDD? In our *sample* they do:

In [ ]:
df_clean.groupby("sex")["LEDD"].agg(["count", "mean", "median"]).round(1)

But a sample is only a small window on the *population*. If we had recruited a different set of patients, the difference might shrink, grow or flip. A hypothesis test asks: **could a difference this large easily arise by chance?**

1. Assume the **null hypothesis (H₀)**: there is no real difference.
2. Work out how big the differences produced by chance alone usually are. This is the **null distribution**.
3. The **p-value** is the proportion of those chance results that are at least as extreme as what we observed.

We can build a null distribution ourselves. If sex made no difference to LEDD, shuffling the sex labels shouldn't matter, so we shuffle them 1,000 times and record the difference each time:

In [ ]:
d = df_clean.dropna(subset=["LEDD", "sex"])
means = d.groupby("sex")["LEDD"].mean()
observed = means["Male"] - means["Female"]

rng = np.random.default_rng(42)   # random number generator; the seed (42) makes the shuffles reproducible
null_diffs = []
for _ in range(1000):
    shuffled_sex = rng.permutation(d["sex"].to_numpy())   # randomly reorder the sex labels
    m = d["LEDD"].groupby(shuffled_sex).mean()   # group means using the shuffled labels
    null_diffs.append(m["Male"] - m["Female"])
null_diffs = np.array(null_diffs)

p_perm = (np.abs(null_diffs) >= abs(observed)).mean()   # share of shuffles at least as extreme as the real difference (either direction)
print(f"Observed difference (Male − Female): {observed:.1f} mg/day")
print(f"Permutation test: {fmt_p(p_perm)}")

plt.hist(null_diffs, bins=40, edgecolor="black", label="Shuffled labels (chance alone)")
plt.axvline(observed, color="red", linewidth=2, label="Observed difference")
plt.xlabel("Mean LEDD difference, Male − Female (mg/day)")
plt.legend()
plt.show()

The observed difference lies far outside anything that shuffling produced, so chance is an unlikely explanation and *p* is tiny. Every test in this notebook works the same way; it just uses a formula instead of shuffling.

**Reading the result**
- By convention, *p* < 0.05 counts as **statistically significant**: we reject H₀. The 0.05 cut-off (called α) is an arbitrary convention, not a law.
- A small *p* does **not** mean the effect is large or clinically important. With thousands of patients, tiny differences become significant.
- *p* = 0.03 does **not** mean a 3% chance that H₀ is true. It means that *if* H₀ were true, a result this extreme would turn up 3% of the time.
- So always report the **effect size** (e.g. the difference in means, in mg/day) with its **95% confidence interval (CI)**: the range of plausible values for the true effect. A 95% CI that excludes 0 matches *p* < 0.05.
- Tests are **two-sided** by default: they look for a difference in either direction. Use a one-sided test only if you predicted the direction *before* seeing the data.

#### <span style="color:red"> 🍎 Exercise 1: your turn! </span>

<span style="color:red">

Repeat the permutation test for **BMI** by sex. Is the difference larger than chance would produce? *Hint: copy the cell above and change one column name.*

</span>

In [ ]:
# Write your code here:

### ✅ Checkpoint 1

Before moving on, check that you can:
- state the null hypothesis for a group comparison;
- explain a p-value in your own words, using the shuffling idea;
- explain why a significant result still needs an effect size and a confidence interval.

---
## 2. Choosing the right test

Most questions have a **parametric** test, which compares means and assumes a roughly normal shape, and a **non-parametric** alternative, which compares ranks and makes no shape assumption:

| Question | Parametric | Non-parametric | `scipy.stats` |
| --- | --- | --- | --- |
| Do **2 groups** differ? | Welch's t-test | Mann–Whitney U | `ttest_ind`, `mannwhitneyu` |
| Do **3+ groups** differ? | One-way ANOVA | Kruskal–Wallis | `f_oneway`, `kruskal` |
| Are **2 numeric** variables related? | Pearson correlation | Spearman correlation | `pearsonr`, `spearmanr` |
| Are **2 categorical** variables related? | Chi-squared test | Fisher's exact test (small counts) | `chi2_contingency`, `fisher_exact` |

> Measuring the *same* patients twice (e.g. before and after treatment)? Then the observations are not independent: use the **paired** versions, `ttest_rel` or `wilcoxon`.

### Checking the assumptions

Parametric tests assume:
1. **Independence**: one row per patient (done above).
2. **Approximate normality** within each group.
3. **Similar spread** (variance) across groups. Welch's t-test relaxes this one, which is why it is the safer default.

When the data are clearly skewed or full of outliers, the mean is a poor summary, so use the non-parametric test. **Look first, test second.** A **Q–Q plot** compares your data with a perfect normal distribution: normal data fall along the straight line, while skewed data curve away from it.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for row, col in enumerate(["BMI", "LEDD"]):   # row = 0 for BMI, 1 for LEDD
    x = df_clean[col].dropna()
    x.plot.hist(ax=axes[row, 0], bins=40, edgecolor="black", title=f"{col}: histogram")
    stats.probplot(x, dist="norm", plot=axes[row, 1])   # Q–Q plot against a normal distribution
    axes[row, 1].set_title(f"{col}: Q–Q plot")
plt.tight_layout()
plt.show()

BMI follows the line, so it is approximately normal. LEDD bends upwards at the right: its long right tail makes it skewed.

Formal tests exist too. **Shapiro–Wilk** tests normality and **Levene's test** tests equal variances; for both, *p* < 0.05 means the assumption is violated. With large samples, however, they flag even trivial deviations, so treat them as a supplement to the plots. Here we run Shapiro–Wilk on a random subset of 300 patients:

In [ ]:
subset = df_clean.sample(300, random_state=0)   # normality tests on a subset: with thousands of rows, trivial deviations become "significant"
for col in ["BMI", "LEDD"]:
    print(f"Shapiro–Wilk, {col}: {fmt_p(stats.shapiro(subset[col].dropna()).pvalue)}")

ledd_by_sex = [g.dropna() for _, g in df_clean.groupby("sex")["LEDD"]]
print(f"Levene, LEDD by sex: {fmt_p(stats.levene(*ledd_by_sex).pvalue)}")

> 💡 `df.groupby("sex")["LEDD"]` yields one Series per group, and the list comprehension collects them. The `*` in `stats.levene(*ledd_by_sex)` unpacks the list into separate arguments, so `levene(*[a, b])` is the same as `levene(a, b)`. You'll use this pattern for every multi-group test.

#### <span style="color:red"> 🍎 Exercise 2: your turn! </span>

<span style="color:red">

Check the assumptions for **UPDRS III**: draw its histogram and Q–Q plot, then decide whether you would compare it between groups with a parametric or a non-parametric test.

</span>

In [ ]:
# Write your code here:

### ✅ Checkpoint 2

Before moving on, check that you can:
- pick the parametric test and its non-parametric alternative for a given question;
- read a Q–Q plot and explain why plots matter more than normality tests in large samples;
- explain why the data must have one row per patient.

---
## 3. Comparing groups

### Two groups: Welch's t-test

BMI looked normal, so we compare mean BMI between men and women with a t-test. Student's classic t-test assumes equal variances; **Welch's** version (`equal_var=False`) does not, and loses almost nothing when the variances are equal, so use it by default. The result also gives the **95% CI** for the difference in means.

In [ ]:
bmi_m = df_clean.loc[df_clean["sex"] == "Male", "BMI"].dropna()
bmi_f = df_clean.loc[df_clean["sex"] == "Female", "BMI"].dropna()

res = stats.ttest_ind(bmi_m, bmi_f, equal_var=False)   # equal_var=False → Welch's t-test
ci = res.confidence_interval()   # 95% CI for the difference in means
print(f"Welch's t-test: t = {res.statistic:.2f}, {fmt_p(res.pvalue)}")
print(f"Mean BMI: Male {bmi_m.mean():.2f}, Female {bmi_f.mean():.2f}")
print(f"Difference (M − F): {bmi_m.mean() - bmi_f.mean():.2f} kg/m², 95% CI {ci.low:.2f} to {ci.high:.2f}")

*p* is well above 0.05 and the CI includes 0: there is **no evidence** that BMI differs by sex. That is not the same as proof that BMI is identical; the CI shows the range of differences that are still plausible.

### Two groups: Mann–Whitney U

LEDD is skewed, so we compare it with the rank-based **Mann–Whitney U** test and summarise each group with its **median** rather than its mean.

In [ ]:
ledd_m = df_clean.loc[df_clean["sex"] == "Male", "LEDD"].dropna()
ledd_f = df_clean.loc[df_clean["sex"] == "Female", "LEDD"].dropna()

res = stats.mannwhitneyu(ledd_m, ledd_f)        # two-sided by default
print(f"Mann–Whitney U: U = {res.statistic:.0f}, {fmt_p(res.pvalue)}")
print(f"Median LEDD: Male {ledd_m.median():.0f}, Female {ledd_f.median():.0f} mg/day")

sns.boxplot(data=df_clean, x="sex", y="LEDD")
plt.title(f"LEDD by sex (Mann–Whitney {fmt_p(res.pvalue)})")
plt.show()

### Three or more groups: ANOVA and Kruskal–Wallis

With more than two groups, running every pairwise test inflates false positives (see Section 5). Instead we start with one **omnibus** test that asks "does *any* group differ?". The parametric version is **one-way ANOVA**, and the rank-based version is **Kruskal–Wallis**.

In [ ]:
display(df_clean.groupby("ethnicity")[["updrs_iii", "LEDD"]].agg(["count", "mean", "median"]).round(1))

updrs_by_eth = [g.dropna() for _, g in df_clean.groupby("ethnicity")["updrs_iii"]]
ledd_by_eth = [g.dropna() for _, g in df_clean.groupby("ethnicity")["LEDD"]]

anova = stats.f_oneway(*updrs_by_eth)   # * unpacks the list: one argument per group
kw = stats.kruskal(*ledd_by_eth)
print(f"ANOVA, UPDRS III by ethnicity:  F = {anova.statistic:.2f}, {fmt_p(anova.pvalue)}")
print(f"Kruskal–Wallis, LEDD by ethnicity: H = {kw.statistic:.2f}, {fmt_p(kw.pvalue)}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(data=df_clean, x="ethnicity", y="updrs_iii", ax=axes[0])
sns.boxplot(data=df_clean, x="ethnicity", y="LEDD", ax=axes[1])
axes[0].set_title("UPDRS III by ethnicity (ANOVA)")
axes[1].set_title("LEDD by ethnicity (Kruskal–Wallis)")
plt.tight_layout()
plt.show()

A significant omnibus test says that *at least one* group differs, not which one. To find out, run pairwise tests and correct for multiple comparisons, as in Section 5.

#### <span style="color:red"> 🍎 Exercise 3: your turn! </span>

<span style="color:red">

For each question, choose a test, justify the choice in one line, run it, and report the result:
1. Does **UPDRS III** differ between men and women?
2. Does **LEDD** differ between patients younger than 60 and those aged 60+? *Hint: create the groups with a boolean mask or `pd.cut`.*
3. Does **BMI** differ across ethnicities?

</span>

In [ ]:
# Write your code here:

### ✅ Checkpoint 3

Before moving on, check that you can:
- run Welch's t-test and Mann–Whitney U, and report a mean difference with its 95% CI or the group medians;
- run ANOVA or Kruskal–Wallis on groups built with `groupby` and `*` unpacking;
- explain what a non-significant result does, and doesn't, tell you.

---
## 4. Associations

### Two numeric variables: correlation

A correlation coefficient (*r*) runs from −1 to +1. The sign gives the direction, and the size (|*r*|) gives the strength: roughly 0.1 is small, 0.3 moderate and 0.5 large.
- **Pearson** measures a *straight-line* relationship. It is sensitive to outliers.
- **Spearman** works on ranks, so it captures any relationship that consistently rises or falls (*monotonic*), even a curved one, and is robust to outliers.

> ⚠️ **Correlation is not causation.** Ice cream sales and drownings rise together because hot weather drives both. A hidden third variable like this (a **confounder**) can create an association between two variables that don't affect each other.

Always plot first:

In [ ]:
d = df_clean.dropna(subset=["years_since_dx", "updrs_iii"])
pear = stats.pearsonr(d["years_since_dx"], d["updrs_iii"])
spear = stats.spearmanr(d["years_since_dx"], d["updrs_iii"])
ci = pear.confidence_interval()   # 95% CI for r

print(f"Pearson r = {pear.statistic:.2f} (95% CI {ci.low:.2f} to {ci.high:.2f}), {fmt_p(pear.pvalue)}")
print(f"Spearman ρ = {spear.statistic:.2f}, {fmt_p(spear.pvalue)}")

# scatter plot plus a fitted straight line with its 95% confidence band
sns.regplot(data=d, x="years_since_dx", y="updrs_iii", scatter_kws={"alpha": 0.2, "s": 10}, line_kws={"color": "red"})
plt.title("UPDRS III vs years since diagnosis")
plt.show()

Here the two coefficients agree: a moderate-to-strong positive association, as expected for a progressive disease.

To scan several variables at once, `DataFrame.corr()` builds a correlation matrix that can be drawn as a heatmap. It gives coefficients only, not *p*-values.

In [ ]:
num_cols = ["age", "BMI", "years_since_dx", "LEDD", "updrs_iii"]
corr = df_clean[num_cols].corr(method="spearman")   # all pairwise correlations (coefficients only, no p-values)

# vmin/vmax fix the colour scale to the full −1 to +1 range
sns.heatmap(corr, annot=True, fmt=".2f", cmap="vlag", vmin=-1, vmax=1, square=True)
plt.title("Spearman correlation matrix")
plt.show()

### Two categorical variables: the chi-squared test

The **chi-squared (χ²) test** compares the counts in a `pd.crosstab` with the counts we would *expect* if the two variables were unrelated. Its assumptions are that each patient appears once and that expected counts are at least 5 in most cells; for a 2×2 table with smaller counts, use `stats.fisher_exact`.

Is severe disease (UPDRS III ≥ 59) associated with sex?

In [ ]:
d = df_clean.dropna(subset=["sex", "updrs_iii"])
# columns are False/True for "UPDRS III ≥ 59"; rename gives them readable labels
table = pd.crosstab(d["sex"], d["updrs_iii"] >= 59).rename(columns={False: "Not severe", True: "Severe"})
display(table)

res = stats.chi2_contingency(table)   # also returns the expected counts (res.expected_freq)
print(f"χ² = {res.statistic:.2f}, df = {res.dof}, {fmt_p(res.pvalue)}")
print("\nExpected counts if sex and severity were unrelated:")
display(pd.DataFrame(res.expected_freq, index=table.index, columns=table.columns).round(1))

Compare the observed and expected tables: the cells that differ most show where the association comes from.

#### <span style="color:red"> 🍎 Exercise 4: your turn! </span>

<span style="color:red">

1. Measure the association between **age** and **LEDD** with both Pearson and Spearman, and plot it. Do the two coefficients agree? Why might they differ?
2. A LEDD above 1000 mg/day is sometimes used to mark *advanced* Parkinson's. Create this binary variable and test whether it is associated with **sex** and with **ethnicity**.

</span>

In [ ]:
# Write your code here:

### ✅ Checkpoint 4

Before moving on, check that you can:
- choose between Pearson and Spearman, and run either with its p-value;
- build a correlation matrix with `.corr()` and draw it as a heatmap;
- run a chi-squared test on a `pd.crosstab` and check its expected counts;
- explain why correlation is not causation.

---
## 5. Multiple comparisons

Every test at α = 0.05 carries a 5% risk of a false positive. Run 20 tests where nothing is going on and you should *expect* one "significant" result by chance. A correlation matrix of 10 variables already contains 45 tests.

Two standard corrections adjust the *p*-values:

| Method | Controls | Use when |
| --- | --- | --- |
| **Bonferroni** | the chance of *any* false positive | a few pre-planned tests; very strict |
| **Benjamini–Hochberg (BH)** | the *proportion* of false positives among significant results (the false discovery rate) | many tests or exploratory screens; more powerful |

Here, the Kruskal–Wallis test told us that LEDD differs somewhere across ethnicities. To find *which* pairs differ, we test every pair and then correct:

In [ ]:
import itertools
from statsmodels.stats.multitest import multipletests

rows = []
# every pair of ethnicities exactly once: Asian vs Black, Asian vs Other, ...
for a, b in itertools.combinations(sorted(df_clean["ethnicity"].dropna().unique()), 2):
    x = df_clean.loc[df_clean["ethnicity"] == a, "LEDD"].dropna()
    y = df_clean.loc[df_clean["ethnicity"] == b, "LEDD"].dropna()
    rows.append({"pair": f"{a} vs {b}", "p": stats.mannwhitneyu(x, y).pvalue})

pairs = pd.DataFrame(rows)
pairs["p_bonferroni"] = multipletests(pairs["p"], method="bonferroni")[1]   # [1] = the adjusted p-values
pairs["p_BH"] = multipletests(pairs["p"], method="fdr_bh")[1]
pairs.round(4)

Compare the columns: some pairs are significant before correction, and still pass with BH, but fail the stricter Bonferroni correction. Decide which correction you will use *before* looking at the results, and report it.

### ✅ Checkpoint 5

Before moving on, check that you can:
- explain why running many tests produces false positives;
- choose between Bonferroni and Benjamini–Hochberg;
- loop over pairs with `itertools.combinations` and adjust the p-values with `multipletests`.

---
## 📝 Reporting your results

State the **question, the test, the statistic, the *p*-value, and an effect with its uncertainty**, and add a line on the assumptions you checked:

* *Welch's t-test:* "Mean BMI did not differ between men and women (Welch's t = …, p = …; difference = … kg/m², 95% CI … to …)."
* *Mann–Whitney U:* "LEDD was higher in men than in women (median … vs … mg/day; Mann–Whitney U = …, p = …)."
* *ANOVA / Kruskal–Wallis:* "LEDD differed across ethnicities (Kruskal–Wallis H = …, p = …); pairwise tests with BH correction showed …"
* *Correlation:* "UPDRS III increased with disease duration (Spearman ρ = …, p = …, n = …)."
* *Chi-squared:* "Severe disease was associated with sex (χ²(1) = …, p = …)."

#### <span style="color:red"> 🍎 Exercise 5: your turn! </span>

<span style="color:red">

**Mini analysis.** Do patients with obesity (BMI ≥ 30) have more severe motor symptoms (UPDRS III)?
1. Decide whether to treat UPDRS III as numeric or as severe / not severe, and choose a test to match.
2. Check the assumptions, run the test, and write a one-sentence report using the templates above.

</span>

In [ ]:
# Write your code here:

Your report:

---
## 📚 Going further (optional)

Topics that build on this notebook but are beyond the scope of this module:
* **Standardised effect sizes**, e.g. Cohen's *d*, η² and rank-biserial *r*: [Learning Statistics with Python, chapters on t-tests and ANOVA](https://ethanweed.github.io/pythonbook/landingpage.html)
* **Post-hoc tests after ANOVA**, e.g. Tukey's HSD: [`statsmodels` `pairwise_tukeyhsd`](https://www.statsmodels.org/stable/generated/statsmodels.stats.multicomp.pairwise_tukeyhsd.html)
* **Risk ratios and odds ratios** for binary outcomes: [`statsmodels` `Table2x2`](https://www.statsmodels.org/stable/generated/statsmodels.stats.contingency_tables.Table2x2.html)
* **Repeated measures**, using every visit rather than one per patient: mixed-effects models, covered in later sessions